In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from rdkit.Chem import MolFromSmiles, GetMolFrags, MolToSmiles
from rdkit.Chem.Draw import MolsToGridImage

import sys
sys.path.append('../')
import FragShapley

/home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
plt.style.use('style.mplstyle')
sns.set_context('talk', font_scale=1.0)
fig_folder = os.path.abspath("figures/")

In [3]:
model = 'GCN'
split = 0


df_expl = pd.read_pickle(f'../4_mutagenicity/{model.lower()}_classification_hansen/df_explanation.pkl')
df_expl = df_expl.loc[df_expl.split == split]
df_expl.head()

,model,dataset,split,smiles,y_true,y_pred,y_pred_proba,fragExplainer_result,fragExplainer_expected_value,frag_to_atom_ids
0,GCN,Hansen,0,O=C(O)c1cc([N+](=O)[O-])cc2cccnc12,1,1.0,0.899424,"{0: -4.898705959320068, 1: 7.668901085853577}",0.359079,"{0: [0, 1, 2], 1: [3, 4, 5, 6, 7, 8, 9, 10, 11..."
1,GCN,Hansen,0,Cc1cc2c(nc(N)n2C)c2ncc(-c3ccccc3)nc12,1,1.0,0.855699,"{0: 3.9027034044265747, 1: -1.5433303713798523}",0.359079,"{0: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,..."
2,GCN,Hansen,0,CCN(N=O)/C(C=O)=N/O,1,1.0,0.987764,{0: 4.970329761505127},0.359079,"{0: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]}"
3,GCN,Hansen,0,C=CC(=O)NCNC(=O)C=C,1,0.0,0.240282,"{0: 0.7583920756975809, 1: -2.0885531306266785...",0.359079,"{0: [0, 1, 2, 3], 1: [4, 5, 6], 2: [7, 8, 9, 10]}"
4,GCN,Hansen,0,Cc1ccccc1-c1ccccc1,0,0.0,0.092938,"{0: -0.33955955505371094, 1: -1.3593494892120361}",0.359079,"{0: [0, 1, 2, 3, 4, 5, 6], 1: [7, 8, 9, 10, 11..."


In [4]:
if not "fragExplainer_expected_value_logits" in df_expl.columns:
    df_expl["fragExplainer_expected_value_logits"] = df_expl.fragExplainer_expected_value.apply(FragShapley.fragshapley.proba_to_logit)

# 1. Obtain List of Non-Toxic Fragments

In [5]:
df_expl['fragments'] = df_expl.smiles.apply(FragShapley.utils.get_BRICS_fragments_as_SMILES)
# get the values of the Explainer as a list (currently only available as dict)
df_expl['fragExplainer_shapley_values'] = df_expl.fragExplainer_result.apply(lambda x: list(x.values()))
# create a dataframe of all fragments (contains duplicates) and the corresponding Shapley Value
df_fragments = df_expl[['fragments', 'model', 'fragExplainer_shapley_values']].explode(['fragments', 'fragExplainer_shapley_values'], ignore_index=True)

In [6]:
df_fragments = df_fragments.groupby(['fragments', 'model']).agg([len, 'mean', 'std', list]) # will throw warning veacuse of error when calculating std for a single measurement
df_fragments = df_fragments.reset_index() # reset so that 'fragments' is a column again and no longer the index
df_fragments.columns = [col[0] if col[1]=='' else col[1] for col in df_fragments.columns.values] # rename, choose 'fragments' for first column, for the other simply len, mean, std

In [7]:
df_fragments.head()

,fragments,model,len,mean,std,list
0,*=C(C(=O)O)C(Cl)Cl,GCN,1,-0.269279,NaN,[-0.26927947998046875]
1,*=C(C)C,GCN,6,-1.798866,1.299177,"[-0.05723097909774105, -1.0941035126646361, -1..."
2,*=C(C)C#N,GCN,1,-3.456076,NaN,[-3.456075668334961]
3,*=C(C)Cl,GCN,1,-0.225729,NaN,[-0.22572851181030273]
4,*=C(C)n1ccnc1,GCN,1,0.603192,NaN,[0.6031924337148665]


# 2. Choose three predicted molecules for detoxification

In [8]:
df_expl['n_fragments'] = df_expl.fragments.apply(len)
df_expl['n_neg_fragments'] = df_expl.fragExplainer_shapley_values.apply(lambda x: sum([1 for i in x if i < 0]))
df_expl['n_pos_fragments'] = df_expl.fragExplainer_shapley_values.apply(lambda x: sum([1 for i in x if i >= 0]))

In [9]:
df_mols = df_expl.query('y_pred == 1 and y_true == 1 and n_pos_fragments == 1 and n_fragments >= 2')

In [10]:
from rdkit.Chem.BRICS import FindBRICSBonds, BreakBRICSBonds
from rdkit.Chem import molzipFragments
from rdkit.Chem import Mol

def remove_atom_map_num(mol):
    mn = Mol(mol)
    for at in mn.GetAtoms():
        at.SetAtomMapNum(0)
    return mn

def isotope_info_to_atom_map_num(mol):
    for at in mol.GetAtoms():
        if at.GetSymbol() == '*':
            info = at.GetIsotope()
            at.SetIsotope(0)
            at.SetAtomMapNum(info)
    return mol

def replace_single_BRICS_substructure_in_mol(mol, smiles_pattern, smiles_replacement):
    # start by getting the bonds that are about to be cut
    brics_bonds = list(FindBRICSBonds(mol))
    # modify in such a way, that the two atoms belonging to the same broken bond have the same isotope information
    brics_bonds_mod = [(t[0], (f'{i+1}', f'{i+1}')) for i, t in enumerate(brics_bonds)]
    # now we can break the BRICS bonds
    brics_broken = BreakBRICSBonds(mol=mol,
                                bonds=brics_bonds_mod,
                                )
    # now turn all of the isotope information into AtomMapNum information
    brics_broken = isotope_info_to_atom_map_num(brics_broken)
    brics_broken_n = remove_atom_map_num(brics_broken)

    # these fragments contain the correctly labeled fragments
    fragments = GetMolFrags(mol=brics_broken,
                            asMols=True)

    # these fragments contain no map num
    # necessary to check which fragments should be replaced
    fragments_n = GetMolFrags(mol=brics_broken_n,
                            asMols=True)
    smiles_n = [MolToSmiles(f) for f in fragments_n]
    idxs_to_replace = [idx for idx, e in enumerate(smiles_n) if e == smiles_pattern]

    output_mols = []

    for idx_to_replace in idxs_to_replace:
        new_frags = ()
        for idx, frag in enumerate(fragments):
            if idx != idx_to_replace:
                new_frags += (frag, )
            else:
                # need to replace
                frag_repl = MolFromSmiles(smiles_replacement)
                # get correct Atom Map Num
                amn = max([at.GetAtomMapNum() for at in frag.GetAtoms()])
                # set correct Atom Map Num
                for at in frag_repl.GetAtoms():
                    if at.GetSymbol() == '*':
                        at.SetAtomMapNum(amn)
                new_frags += (frag_repl, )
        try:
            output_mols.append(molzipFragments(new_frags))
        except:
            pass
        
    return output_mols

In [11]:
rows_of_interest = {'GCN': [3, 28, 35, 38, 54]}
replacement_id = {'GCN': [1, 6, 9, 7, 13]}

In [12]:
# extract data from row

old_smiles = []
replaced_frags = []
new_frags = []
new_smiles = []


for index, replacement_idx in zip(rows_of_interest[model], replacement_id[model]):
    row = df_mols.iloc[index]

    smiles = row.smiles
    y_pred_proba = row.y_pred_proba
    explainer_result = row.fragExplainer_result
    expected_value = row.fragExplainer_expected_value
    frag_to_atom_ids = row.frag_to_atom_ids

    # get list of replaced molecules
    negative_id = [i for i, v in explainer_result.items() if v > 0]
    assert len(negative_id) == 1
    negative_id = negative_id[0]
    frag_to_replace = row.fragments[negative_id]

    n_attachment = frag_to_replace.count('*')

    df_fragments['n_attachments'] = df_fragments.fragments.apply(lambda x: x.count('*'))
    df_tmp = df_fragments.loc[df_fragments.n_attachments == n_attachment]
    # get possible replacement fragments
    df_tmp = df_tmp.loc[df_tmp['mean'] <= 0.0]
    df_tmp = df_tmp.loc[df_tmp.len >= 3]
    df_tmp['mean_plus_one_std'] = df_tmp['mean'] + df_tmp['std']
    new_frag = df_tmp.sort_values(by='mean_plus_one_std').iloc[replacement_idx].fragments

    out_mols = replace_single_BRICS_substructure_in_mol(MolFromSmiles(row.smiles), frag_to_replace, new_frag)

    old_smiles.append(smiles)
    replaced_frags.append(frag_to_replace)
    new_frags.append(new_frag)
    new_smiles.append(MolToSmiles(out_mols[0]))
    
df = pd.DataFrame({'old_smiles': old_smiles,
                   'replaced_frags': replaced_frags,
                   'new_frags': new_frags,
                   'new_smiles': new_smiles,})

In [13]:
df

,old_smiles,replaced_frags,new_frags,new_smiles
0,O=C(NO)c1ccccc1O,*C(=O)NO,*C(F)(F)F,Oc1ccccc1C(F)(F)F
1,[N-]=[N+]=NCc1ccccc1,*CN=[N+]=[N-],*CCCC,CCCCc1ccccc1
2,c1ccc([C@H]2CO2)cc1,*[C@H]1CO1,*C1CC1,c1ccc(C2CC2)cc1
3,CN(Cc1ccccc1)N=O,*CN(C)N=O,*CC(C)C,CC(C)Cc1ccccc1
4,O=C(Cl)Cc1ccccc1,*CC(=O)Cl,*CC(=O)O,O=C(O)Cc1ccccc1


In [14]:
# load model

import torch
import lightning as L
from torch.utils.data import DataLoader
import numpy as np

if model in ['GCN', 'MPNN']:
    classifier = torch.load(f'../4_mutagenicity/{model.lower()}_classification_hansen/models/model_{model.lower()}c_Hansen_split_{split}.pkl', weights_only=False)
    featurizer = FragShapley.Featurizer(input_format='smiles',
                                        output_format='graph')
    # get new predictions
    ds_old = FragShapley.MoleculeDataset(list_of_inputs=df.old_smiles,
                                            featurizer=featurizer)
    ds_new = FragShapley.MoleculeDataset(list_of_inputs=df.new_smiles,
                                            featurizer=featurizer)
    trainer = L.Trainer(accelerator='auto')
    pred_old = trainer.predict(model=classifier, dataloaders=ds_old)
    pred_new = trainer.predict(model=classifier, dataloaders=ds_new)
    
    df['prediction_old'] = np.array(pred_old).squeeze()
    df['prediction_new'] = np.array(pred_new).squeeze()

    frag_explainer = FragShapley.FragmentExplainer(model=classifier,
                                                   expected_value='empty',
                                                   fragmentation_method='BRICS',
                                                   representation='graph',
                                                   trainer=trainer,
                                                   featurizer=featurizer,
                                                   batch_size=8,)
    ev_frag = frag_explainer.expected_value
    results_dict_new, frag_to_atom_ids_new = frag_explainer.explain(df.new_smiles.to_list(),
                                                                    return_atom_id_to_bits=False,)
    df['explainer_new'] = results_dict_new
    df['frag_to_atom_ids_new'] = frag_to_atom_ids_new
    df['ev_new'] = ev_frag

    results_dict_old, frag_to_atom_ids_old = frag_explainer.explain(df.old_smiles.to_list(),
                                                                    return_atom_id_to_bits=False,)
    df['explainer_old'] = results_dict_old
    df['frag_to_atom_ids_old'] = frag_to_atom_ids_old
    df['ev_old'] = ev_frag

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
/home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


Predicting DataLoader 0: 100%|██████████| 5/5 [00:00<00:00, 377.28it/s]

/home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.



Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 211.00it/s]


/home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 239.01it/s]


In [15]:
df

,old_smiles,replaced_frags,new_frags,new_smiles,prediction_old,prediction_new,explainer_new,frag_to_atom_ids_new,ev_new,explainer_old,frag_to_atom_ids_old,ev_old
0,O=C(NO)c1ccccc1O,*C(=O)NO,*C(F)(F)F,Oc1ccccc1C(F)(F)F,0.699488,0.013371,"{0: 2.582561671733856, 1: -6.304321050643921}","{0: [0, 1, 2, 3, 4, 5, 6], 1: [7, 8, 9, 10]}",0.359079,"{0: 1.7170850038528442, 1: -0.2928641140460968}","{0: [0, 1, 2, 3], 1: [4, 5, 6, 7, 8, 9, 10]}",0.359079
1,[N-]=[N+]=NCc1ccccc1,*CN=[N+]=[N-],*CCCC,CCCCc1ccccc1,0.854864,0.002388,"{0: -4.5825183391571045, 1: -0.8726290464401245}","{0: [0, 1, 2, 3], 1: [4, 5, 6, 7, 8, 9]}",0.359079,"{0: 3.317420184612274, 1: -0.9647912383079529}","{0: [0, 1, 2, 3], 1: [4, 5, 6, 7, 8, 9]}",0.359079
2,c1ccc([C@H]2CO2)cc1,*[C@H]1CO1,*C1CC1,c1ccc(C2CC2)cc1,0.992134,0.011864,"{0: -1.724050760269165, 1: -2.1187827587127686}","{0: [0, 1, 2, 3, 7, 8], 1: [4, 5, 6]}",0.359079,"{0: -6.147709846496582, 1: 11.564267635345459}","{0: [0, 1, 2, 3, 7, 8], 1: [4, 5, 6]}",0.359079
3,CN(Cc1ccccc1)N=O,*CN(C)N=O,*CC(C)C,CC(C)Cc1ccccc1,0.966987,0.003053,"{0: -4.209198474884033, 1: -0.9996458292007446}","{0: [0, 1, 2, 3], 1: [4, 5, 6, 7, 8, 9]}",0.359079,"{0: 8.587396621704102, 1: -4.630776524543762}","{0: [0, 1, 2, 9, 10], 1: [3, 4, 5, 6, 7, 8]}",0.359079
4,O=C(Cl)Cc1ccccc1,*CC(=O)Cl,*CC(=O)O,O=C(O)Cc1ccccc1,0.907123,0.000431,"{0: -3.2003369331359863, 1: -3.9660526514053345}","{0: [0, 1, 2, 3], 1: [4, 5, 6, 7, 8, 9]}",0.359079,"{0: 4.068526268005371, 1: -1.2101657390594482}","{0: [0, 1, 2, 3], 1: [4, 5, 6, 7, 8, 9]}",0.359079


In [ ]:
from rdkit.Chem import rdFMCS, MolFromSmarts, AllChem

for i, r in df.iterrows():
    smiles_old, smiles_new = r.old_smiles, r.new_smiles
    proba_old, proba_new = r.prediction_old, r.prediction_new
    expl_old, expl_new = r.explainer_old, r.explainer_new
    ftai_old, ftai_new = r.frag_to_atom_ids_old, r.frag_to_atom_ids_new

    mol_old, mol_new = MolFromSmiles(smiles_old), MolFromSmiles(smiles_new)
    mcs_result = rdFMCS.FindMCS([mol_old, mol_new],
                                ringMatchesRingOnly=True,
                                completeRingsOnly=True,
                                )
    mcs_smarts = mcs_result.smartsString
    mcs_mol = MolFromSmarts(mcs_smarts)

    AllChem.Compute2DCoords(mol_old)
    AllChem.GenerateDepictionMatching2DStructure(mol_new, mol_old, refPatt=mcs_mol)

    contr_old = FragShapley.visualization.get_atom_contribution_from_result_dict(smiles=smiles_old,
                                                                                 results_dict=expl_old,
                                                                                 frag_to_atom_ids=ftai_old)
    contr_new = FragShapley.visualization.get_atom_contribution_from_result_dict(smiles=smiles_new,
                                                                                 results_dict=expl_new,
                                                                                 frag_to_atom_ids=ftai_new)
    svg_old = FragShapley.visualization.visualize_contributions_from_mol_new(mol=mol_old,
                                                                             contributions=expl_old.values(),
                                                                             frag_to_atom_ids=ftai_old,
                                                                             min=-12,
                                                                             max=12,
                                                                             colormap='bwr')
    svg_new = FragShapley.visualization.visualize_contributions_from_mol_new(mol=mol_new,
                                                                             contributions=expl_new.values(),
                                                                             frag_to_atom_ids=ftai_new,
                                                                             min=-12,
                                                                             max=12,
                                                                             colormap='bwr')
    print(f"MIN: {min(contr_old)}, MAX: {max(contr_old)}")
    print(f"MIN: {min(contr_new)}, MAX: {max(contr_new)}")
    with open(os.path.join(fig_folder, f"molecules_mutagenicity/2_mutagenicity_optimization_{i}_old.svg"), "w") as f:
        f.write(svg_old.data)
    with open(os.path.join(fig_folder, f"molecules_mutagenicity/2_mutagenicity_optimization_{i}_new.svg"), "w") as f:
        f.write(svg_new.data)

MIN: -0.2928641140460968, MAX: 1.7170850038528442
MIN: -6.304321050643921, MAX: 2.582561671733856
MIN: -0.9647912383079529, MAX: 3.317420184612274
MIN: -4.5825183391571045, MAX: -0.8726290464401245
MIN: -6.147709846496582, MAX: 11.564267635345459
MIN: -2.1187827587127686, MAX: -1.724050760269165
MIN: -4.630776524543762, MAX: 8.587396621704102
MIN: -4.209198474884033, MAX: -0.9996458292007446
MIN: -1.2101657390594482, MAX: 4.068526268005371
MIN: -3.9660526514053345, MAX: -3.2003369331359863


In [17]:
contr_new

array([-3.20033693, -3.20033693, -3.20033693, -3.20033693, -3.96605265,
       -3.96605265, -3.96605265, -3.96605265, -3.96605265, -3.96605265])